In [0]:
# Loading Bronze tables
customers = spark.read.table("revenue_operations.bronze.customers")
products = spark.read.table("revenue_operations.bronze.products")
sellers = spark.read.table("revenue_operations.bronze.sellers")
orders = spark.read.table("revenue_operations.bronze.orders")
order_items = spark.read.table("revenue_operations.bronze.order_items")
product_category_translation = spark.read.table("revenue_operations.bronze.product_category_translation")

### Silver Customers

In [0]:
customers.show(5)
customers.printSchema()

In [0]:
from pyspark.sql.functions import column 
# Duplicate Check
duplicate_customers = customers.count() - customers.dropDuplicates().count()
print(f"Number of duplicate rows : {duplicate_customers}")
print("\nTotal rows in customers:", customers.count())

# Checking for null values
for col in customers.columns:
    print(f"Number of null values in {col}: {customers.filter(column(col).isNull()).count()}")

print("\n")
# Checking for Distinct values
for col in customers.columns:
    print(f"Number of distinct values in {col}: {customers.select(col).distinct().count()}")

There are 96096 total unique individual customers. The 99441 in the customer_id is connecting to the customer_id in the orders table where each unique customer get new customer_id for each order. So, customer_unique_id can be used to identify a single customer orders and their purchase behavior.

In [0]:
# Foreign keys check
print(f"Total number of customer_id in customer dataset : {customers.select("customer_id").distinct().count()}")
print(f"Total number of customer_id in orders dataset: {orders.select("customer_id").distinct().count()}")

# Checking if customers in orders exist in customers
missing_customers = orders.join(customers.select("customer_id"), on = "customer_id", how = "leftanti").count()
print("Number of customers in customers that does not exist in orders = ", missing_customers)

Customers Silver Design Notes

Source = revenue_operations.bronze.customers

Target = revenue_operations.silver.customers

Columns renamed = None

Data type Changes = None

Duplicate findings = None (No duplicate rows)

Null findings = None

Primary Key = customer_id

Validation findings: None (No validations needed other than checking for null values and checking foreign keys.)

Foreign Key findings = All the customer_id in the orders dataset are in customer dataset(Parent).

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status

#### Silver Customers Dataframe

In [0]:
silver_customers = customers.select("*")

# Dropping duplicate rows
print("Before: ", silver_customers.count())
silver_customers = silver_customers.dropDuplicates()
print("After: ", silver_customers.count())

from pyspark.sql.functions import lit
from pyspark.sql import functions as F
# Adding audit columns
audit_columns = ['source_file_name', 'ingestion_timestamp', 'silver_processed_timestamp', 'data_quality_status']
silver_customers = silver_customers.withColumn('source_file_name', lit("olist_customer_dataset.csv")).withColumn('ingestion_timestamp', F.expr("current_timestamp()")).withColumn('silver_processed_timestamp', F.expr("current_timestamp()")).withColumn('data_quality_status', lit("valid"))

silver_customers.groupBy("data_quality_status").count().orderBy("count", ascending=False).show() 


In [0]:
# Writing silver customers table
silver_customers.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.customers")

In [0]:
silver_customers_table = spark.read.table("revenue_operations.silver.customers")
print("Total rows: ",silver_customers_table.count())
silver_customers_table.printSchema()

### Silver products

In [0]:
products.show(5)
products.printSchema()

In [0]:
# Duplicate Check
duplicate_products = products.count() - products.dropDuplicates().count()
print("Number of duplicate rows : ", duplicate_products)

print("Total rows in products =", products.count())
# Checking null values
for col in products.columns:
    print(f"Null values in {col}: {products.filter(column(col).isNull()).count()}")

# Distinct Values Count in each columns
for col in products.columns:
    print(f"Number of distinct values in {col}: {products.select(col).distinct().count()}")

In [0]:
# Foreign key check
# Checking for product_id that exist in order_items but do not exist in products
missing_products = order_items.select("product_id").join(products.select("product_id"), on="product_id", how="leftanti").count()
print(f"The number of product_id that exist in order_items but do not exist in products = {missing_products}")

# Checking for product_category_name in products that do not exist in product_category_translation (parent table)
missing_product_cat_unique = products.select("product_category_name").join(product_category_translation.select("product_category_name"), on="product_category_name", how="leftanti").distinct()

print(f"The number of UNIQUE missing categories in products_category_translation = {missing_product_cat_unique.count()}")
missing_product_cat_unique.show()
print("Products dataset unique categories = ", products.select("product_category_name").distinct().count())
print("Product Category translation unique categories = ", product_category_translation.select("product_category_name").distinct().count())


Products Silver Design Notes

Source = revenue_operations.bronze.products

Target = revenue_operations.silver.products

Columns renamed = None

Data type Changes = None

Duplicate findings = None (No duplicate rows)

Null findings = None

Primary Key = product_id

Validation findings: None (No validations needed other than checking for null values and checking foreign keys.)

Foreign Key findings = Products has 2 extra product_category_name than its parent product_name_category_translation table (3 but NULL is not a category name, it is missing entry.) The 2 missing category_names will not be flagged as one of them is a coherent english category, and second one has no translation, instead of manually translation, we will let the category be as is.

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status

#### Silver products dataframe

In [0]:
from pyspark.sql.functions import lit, current_timestamp, when

silver_products = products.select("*")

print("Dropping Duplicate Rows")
print("Before: ", silver_products.count())
silver_products = silver_products.dropDuplicates()
print("After: ", silver_products.count())

silver_products = silver_products.withColumn('source_file_name', lit("olist_products_dataset.csv")) \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('silver_processed_timestamp', current_timestamp()) \
    .withColumn('data_quality_status', when(column("product_category_name").isNull() | column("product_name_lenght").isNull() | column("product_description_lenght").isNull() | column("product_photos_qty").isNull() | column("product_weight_g").isNull() | column("product_length_cm").isNull() | column("product_height_cm").isNull() | column("product_width_cm").isNull(), lit("missing value")).otherwise(lit("valid")))

display(silver_products.groupBy("data_quality_status").count().orderBy("count", ascending=False))

In [0]:
# Writing Silver Products table
silver_products.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.products")

In [0]:
silver_products_table = spark.read.table("revenue_operations.silver.products")
print("Total rows: ",silver_products_table.count())
silver_products_table.printSchema()

### Silver Sellers

In [0]:
sellers.show(5)
sellers.printSchema()

In [0]:
print("Total Number of sellers = ", sellers.select("seller_id").distinct().count())
print("Total Duplicate rows = ", sellers.count() - sellers.dropDuplicates().count())

# Null Values in sellers
for col in sellers.columns:
    print(f"Null Values in {col}: {sellers.filter(column(col).isNull()).count()}")

# Distinct Values in sellers
for col in sellers.columns:
    print(f"Distinct values in {col}: {sellers.select(col).distinct().count()}")

In [0]:
# Foreign key check
# Number of seller_id that are in order_items but not in sellers
missing_seller = order_items.select("seller_id").join(sellers.select("seller_id"), on="seller_id", how="leftanti").count()
print(f"The number of seller_id that are in order_items but not in sellers = {missing_seller}")

Sellers Silver Design Notes

Source = revenue_operations.bronze.sellers

Target = revenue_operations.silver.sellers

Columns renamed = None

Data type Changes = None

Duplicate findings = None (No duplicate rows)

Null findings = None

Primary Key = seller_id

Validation findings: None (No validations needed other than checking for null values and checking foreign keys.)

Foreign Key findings = ALl foreign key good

Audit columns = source_file_name, ingestion_timestamp, silver_processed_timestamp, data_quality_status

#### Silver sellers dataframe

In [0]:
from pyspark.sql.functions import lit, current_timestamp

silver_sellers = sellers.select("*")

print("Dropping Duplicate Rows")
print("Before: ", silver_sellers.count())
silver_sellers = silver_sellers.dropDuplicates()
print("After: ", silver_sellers.count())

silver_sellers= silver_sellers.withColumn('source_file_name', lit("olist_sellers_dataset.csv")) \
    .withColumn('ingestion_timestamp', current_timestamp()) \
    .withColumn('silver_processed_timestamp', current_timestamp()) \
    .withColumn('data_quality_status', lit("valid"))

display(silver_sellers.groupBy("data_quality_status").count().orderBy("count", ascending=False))

In [0]:
# Writing silver sellers table
silver_sellers.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.silver.sellers")

In [0]:
silver_sellers_table = spark.read.table("revenue_operations.silver.sellers")
print("Total rows: ",silver_sellers_table.count())
silver_sellers_table.printSchema()